# Network Anomaly Detection - VNTD

This document trains an **Isolation Forest** model to detect anomalous network traffic from Suricata logs.

### How it works

1. Teach the model what **normal traffic looks like** (using only a benign dataset).
2. Ask the model to evaluate all traffic; anything that looks different from normal is flagged as an anomaly.

### Why train only on benign data?

The `attacks.json` file can contain around 300.000 entries, most of which are DoS flood packets. If we train the model on that, it would think DoS traffic is *normal*. By training only the benign flows (much smaller dataset), the model learns what legitimate traffic look slike; and everything else becomes suspicious.

### Files used

| File                | Description                                                          |
|:--------------------|:---------------------------------------------------------------------|
| `data/benign.json`  | ~200 flows of normal traffic; used to **train** the model            |
| `data/attacks.json` | ~300.000 flows of attack traffic (DoS, port scan, SSH bruteforce...) |
| `data/data.json`    | Combined dataset; used to **evaluate** the model                     |


---

## Step 1 - Load the data

Suricata writes logs in a **JSON** file, in a format where each line represents an object.
We only care about events of type `flow`; these represent a complete network connection and contain the most useful information (bytes sent, packets, duration, protocol...).

We import `json` (to parse each line) and `pandas` (to work with the data as a table).


In [1]:
import json
import pandas as pd

#def flatten_json(d, parent_key='', sep='_'):
#    items = []
#    for k, v in d.items():
#        new_key = f"{parent_key}{sep}{k}" if parent_key else k
#        if isinstance(v, dict):
#            items.extend(flatten_json(v, new_key, sep=sep).items())
#        else:
#            items.append((new_key, v))
#    return dict(items)

def flatten_json(d, parent_key='', sep='_'):
    items = []

    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k

        if isinstance(v, dict):
            items.extend(flatten_json(v, new_key, sep=sep).items())

        elif isinstance(v, list):
            if len(v) == 0:
                items.append((new_key, None))
            else:
                for i, item in enumerate(v):
                    list_key = f"{new_key}{sep}{i}"

                    if isinstance(item, dict):
                        items.extend(flatten_json(item, list_key, sep=sep).items())
                    else:
                        items.append((list_key, item))

        else:
            items.append((new_key, v))

    return dict(items)

def load_flows(file):
    rows = []

    with open(file, "r") as f:
        for line in f:
            try:
                event = json.loads(line.strip())
            except json.JSONDecodeError:
                continue                                    # Skip any lines that can not be processed

            if event.get("event_type") != "flow":           # Only process flow events, no alerts
                continue
                
            row = flatten_json(event)
            rows.append(row)

    return pd.DataFrame(rows)

In [2]:
# Load the file
data = load_flows("../data.json")

# Quick check
print(f"Benign flows: {len(data):>12,}")

Benign flows:      325,181


Event example prior to processing:

```json
{
   "timestamp":"2026-04-10T14:03:15.502440+0000",
   "flow_id":1099387108089453,
   "in_iface":"eth1",
   "event_type":"flow",
   "src_ip":"10.0.0.2",
   "src_port":38018,
   "dest_ip":"192.168.10.10",
   "dest_port":22,
   "proto":"TCP",
   "flow":{
      "pkts_toserver":2,
      "pkts_toclient":0,
      "bytes_toserver":128,
      "bytes_toclient":0,
      "start":"2026-04-10T13:58:36.548461+0000",
      "end":"2026-04-10T13:58:36.549139+0000",
      "age":0,
      "state":"new",
      "reason":"timeout",
      "alerted":false
   },
   "community_id":"1:T/hpCH2HdAaBxpHOZeIqbBSQwqo=",
   "tcp":{
      "tcp_flags":"06",
      "tcp_flags_ts":"06",
      "tcp_flags_tc":"00",
      "syn":true,
      "rst":true,
      "state":"syn_sent"
   }
}
```

---

## Step 2 - Look at the raw data

Before doing anything with ML, it's a good practise to just **look** at what we have.

First, obtain some details from the raw non-processed data:

In [3]:
import json
import pandas as pd

records = []
with open("../data.json", "r") as f:
    for line in f:
        try:
            records.append(json.loads(line.strip()))
        except:
            pass

df = pd.DataFrame(records)
print(f"Total events: {len(df)}")
print(df["event_type"].value_counts())

Total events: 325679
event_type
flow     325181
alert       496
dns           2
Name: count, dtype: int64


In [4]:
# Flow are richer for anomaly detection
flows = df[df["event_type"] == "flow"].copy()

# Expand the 'flow' field, a nested dictionary
flow_details = pd.json_normalize(flows["flow"])
flows = pd.concat([flows.reset_index(drop=True), flow_details], axis=1)

print(flows.shape)
flows[["src_ip","dest_ip","proto","pkts_toserver","pkts_toclient",
       "bytes_toserver","bytes_toclient","age","state"]].head(10)

(325679, 32)


,src_ip,dest_ip,proto,pkts_toserver,pkts_toclient,bytes_toserver,bytes_toclient,age,state
0,fe80:0000:0000:0000:a8c1:abff:fea2:67ad,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,1.0,0.0,70.0,0.0,0.0,new
1,fe80:0000:0000:0000:a8c1:abff:fea2:67ad,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,1.0,0.0,70.0,0.0,0.0,new
2,fe80:0000:0000:0000:a8c1:abff:fe08:60bd,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,1.0,0.0,70.0,0.0,0.0,new
3,fe80:0000:0000:0000:a8c1:abff:fe08:60bd,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,1.0,0.0,70.0,0.0,0.0,new
4,172.16.100.100,192.168.10.10,UDP,1.0,0.0,88.0,0.0,0.0,new
5,172.16.100.100,192.168.10.10,UDP,2.0,0.0,172.0,0.0,0.0,new
6,20.0.0.2,192.168.10.10,TCP,6.0,0.0,482.0,0.0,0.0,new
7,172.16.100.100,192.168.10.10,UDP,1.0,0.0,86.0,0.0,0.0,new
8,172.16.100.100,192.168.10.10,UDP,1.0,0.0,72.0,0.0,0.0,new
9,10.0.0.2,192.168.10.10,TCP,1.0,0.0,54.0,0.0,0.0,new


Now, process the data once we know what type of data is recovered:

In [5]:
data.head()

,timestamp,flow_id,in_iface,event_type,src_ip,dest_ip,proto,icmp_type,icmp_code,flow_pkts_toserver,...,tcp_syn,tcp_fin,tcp_psh,tcp_ack,tcp_state,tcp_rst,tcp_ecn,tcp_cwr,metadata_flowbits_0,metadata_flowbits_1
0,2026-04-10T13:53:08.233561+0000,775976044357742,eth1,flow,fe80:0000:0000:0000:a8c1:abff:fea2:67ad,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,133.0,0.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2026-04-10T13:54:39.479798+0000,775976049198968,eth1,flow,fe80:0000:0000:0000:a8c1:abff:fea2:67ad,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,133.0,0.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2026-04-10T13:55:07.018619+0000,1509668131419117,eth1,flow,fe80:0000:0000:0000:a8c1:abff:fe08:60bd,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,133.0,0.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2026-04-10T13:57:05.200057+0000,1509668140286312,eth1,flow,fe80:0000:0000:0000:a8c1:abff:fe08:60bd,ff02:0000:0000:0000:0000:0000:0000:0002,IPv6-ICMP,133.0,0.0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2026-04-10T13:57:15.228249+0000,1240343632229848,eth1,flow,172.16.100.100,192.168.10.10,UDP,NaN,NaN,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
data.describe()

,flow_id,icmp_type,icmp_code,flow_pkts_toserver,flow_pkts_toclient,flow_bytes_toserver,flow_bytes_toclient,flow_age,src_port,dest_port
count,3.251810e+05,8.0,8.0,325181.000000,325181.0,325181.000000,325181.0,325181.000000,325173.000000,325173.000000
mean,1.124858e+15,133.0,0.0,1.607606,0.0,106.043028,0.0,6.501127,32253.963364,90.238098
std,6.505539e+14,0.0,0.0,1.533247,0.0,103.137814,0.0,14.966267,18932.440964,693.550250
min,4.360826e+09,133.0,0.0,1.000000,0.0,54.000000,0.0,0.000000,0.000000,22.000000
25%,5.608734e+14,133.0,0.0,1.000000,0.0,66.000000,0.0,0.000000,15864.000000,80.000000
50%,1.122498e+15,133.0,0.0,1.000000,0.0,66.000000,0.0,0.000000,31641.000000,80.000000
75%,1.687574e+15,133.0,0.0,1.000000,0.0,66.000000,0.0,0.000000,48678.000000,80.000000
max,2.251793e+15,133.0,0.0,56.000000,0.0,5442.000000,0.0,62.000000,65535.000000,60304.000000


---

## Step 3 - Derived features

The raw numbers are useful, but we can also create new features that make patterns more obvious to detect.

For example:
- A **DoS flood** sends thousands of packets but gets no response -> `pk_ratio` will be high.
- A **port scan** sends small packets to many ports -> `bytes_per_pkt` will be very low.
- Normal traffic is mostly balanced in both directions.

In [11]:
def add_features(df):
    #df = df.copy()                                                    # Copy so the original is not modified, necessary?
    total_pkts = df["flow_pkts_toserver"] + df["flow_pkts_toclient"]
    total_bytes = df["flow_bytes_toserver"] + df["flow_bytes_toclient"]

    # AVG Byte per packet, small value means scan traffic
    df["bytes_per_pkt"] = total_bytes / total_pkts.clip(lower=1)                            # clip(lower=1) replaces all values <1 with 1

    # Ratio of packets in each direction, high value may mean DoS
    df["pkt_ratio"] = df["flow_pkts_toserver"] / df["flow_pkts_toclient"].clip(lower=1)

    # Same, but for bytes
    df["bytes_ratio"] = df["flow_bytes_toserver"] / df["flow_bytes_toclient"].clip(lower=1) 

    return df

# Apply such to both datasets

# IN THE FUTURE ADD THE SECOND ONE
data = add_features(data)

print("New columns added: bytes_per_pkt, pkt_ratio, bytes_ratio")

New columns added: bytes_per_pkt, pkt_ratio, bytes_ratio


---

## Step 4 - Visualize the difference between benign and attack traffic

User `matplotlib` to plot histograms. A histogram shows how often different values appear. If the benign (*blue*) and attack (*red*) distributions look very different -> the model will be able to tell them apart.

In [ ]:
import matplotlib.pyplot as plt

# Features we want to visualise
features_to_plot = [
    "bytes_toserver",
    "pkts_toserver",
    "age",
    "bytes_per_pkt",
    "pkt_ratio",
    "bytes_ratio",
]

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle("Feature distributions: Benign (blue) vs Attacks (red)", fontsize=13)

for ax, feat in zip(axes.flat, features_to_plot):
    # Clip extreme values so the histogram is readable
    # (DoS floods can have millions of bytes -> we cap at the 99th percentile)
    #cap_b = df_benign[feat].quantile(0.99)
    #cap_a = df_attacks[feat].quantile(0.99)
    #cap   = max(cap_b, cap_a)

    vals_b = df_benign[feat]#.clip(upper=cap)
    vals_a = df_attacks[feat]#.clip(upper=cap)

    ax.hist(vals_b, bins=40, alpha=0.6, color="steelblue", label="Benign",  density=True)
    ax.hist(vals_a, bins=40, alpha=0.6, color="tomato",    label="Attacks", density=True)
    ax.set_title(feat)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("../models/feature_distributions.png", dpi=150)
plt.show()

print("Plot saved to ../models/feature_distributions.png")